# Faithfulness-validated hybrid explanations for transformer phishing-email detection
Run all cells on a free Colab T4 (Runtime > Change runtime type > T4 GPU).
Upload `Phishing_Email.csv` (Kaggle: subhajournal/phishingemails) when asked.
Everything the paper reports is produced here. At the end, download `results.zip`
and send it back. Nothing in this notebook is hard-coded or simulated.

SAFE TO INTERRUPT: progress is saved to your Google Drive (folder `phish_xai`).
If Colab disconnects, click Reconnect and choose Runtime > Run all again.
Finished stages print "resumed" and are skipped, so you lose almost nothing.
Tip: put Phishing_Email.csv in Drive/phish_xai/ once and it is never asked for again.

### Setup

In [ ]:
import os, sys, subprocess, json, re, time, random, math, zipfile, warnings, pickle, shutil
warnings.filterwarnings("ignore")

def pip(*p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)

if "google.colab" in sys.modules or os.environ.get("INSTALL") == "1":
    pip("transformers", "accelerate", "lime", "shap", "captum",
        "scikit-learn", "matplotlib", "scipy", "datasets")

import numpy as np, pandas as pd, torch
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

QUICK = os.environ.get("QUICK") == "1"   # QUICK=1 -> tiny run to test the notebook
CFG = dict(
    seeds=[42, 43, 44], max_len=256, epochs=3, lr=2e-5, bs=32,
    models=["distilroberta-base", "roberta-base"],
    seeds_for={"distilroberta-base": [42, 43, 44], "roberta-base": [42]},
    n_expl_test=100, n_expl_val=40, expl_words=120,
    ig_steps=30, lime_samples=500, shap_evals=300,
    csv_path="Phishing_Email.csv", out="results",
    ext_datasets=[("zefang-liu/phishing-email-dataset", None),
                  ("SetFit/enron_spam", None)],
)
if QUICK:
    CFG.update(seeds=[42], epochs=1, models=["distilroberta-base"],
               seeds_for={"distilroberta-base": [42]},
               n_expl_test=6, n_expl_val=4, ig_steps=8, lime_samples=60,
               shap_evals=60, ext_datasets=[])

# Everything is saved to Google Drive so a disconnected runtime never loses progress.
# If the runtime drops: reconnect, Run all again, and finished work is skipped ("resumed").
IN_COLAB = "google.colab" in sys.modules
DRIVE_DIR = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/phish_xai"
    CFG["out"] = DRIVE_DIR + "/" + ("quick" if QUICK else "full") + "/results"
OUT = CFG["out"]
CKPT = os.path.dirname(os.path.abspath(OUT)) + "/ckpt"
os.makedirs(OUT + "/figures", exist_ok=True); os.makedirs(CKPT, exist_ok=True)
def ckf(*p): return os.path.join(CKPT, "_".join(str(x) for x in p))

def sync_drive(tag=""):
    """Force Colab to flush pending writes to Google Drive (otherwise a recycled runtime can lose them)."""
    if not IN_COLAB: return
    try:
        from google.colab import drive as _d
        _d.flush_and_unmount(); _d.mount("/content/drive", force_remount=True)
        n = sum(len(f) for _, _, f in os.walk(CKPT))
        print(f"[drive synced {tag}] files in ckpt: {n}")
    except Exception as e:
        print("[drive sync failed]", e)
print("saving to:", OUT)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

### Helpers: text, metrics, statistics

In [ ]:
WORD = re.compile(r"\w+")

def clean(t):
    t = re.sub(r"<[^>]+>", " ", str(t))
    return re.sub(r"\s+", " ", t).strip()

def key(t):
    return re.sub(r"\W+", " ", str(t).lower()).strip()

def metrics(y, p, prob=None):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    tn = int(((y == 0) & (p == 0)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tp = int(((y == 1) & (p == 1)).sum())
    m = dict(acc=accuracy_score(y, p), prec=pr, rec=rc, f1=f1,
             mcc=matthews_corrcoef(y, p) if len(set(p)) > 1 else 0.0,
             fpr=fp / max(fp + tn, 1), fnr=fn / max(fn + tp, 1),
             tn=tn, fp=fp, fn=fn, tp=tp)
    if prob is not None and len(set(y)) > 1:
        m["auc"] = roc_auc_score(y, prob)
    return m

def bootstrap_ci(y, p, fn, B=1000, seed=0):
    rng = np.random.RandomState(seed); n = len(y); v = []
    for _ in range(B):
        i = rng.randint(0, n, n)
        v.append(fn(y[i], p[i]))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def mcnemar(y, pa, pb):
    a = (pa == y); b = (pb == y)
    n01 = int((a & ~b).sum()); n10 = int((~a & b).sum())
    if n01 + n10 == 0:
        return 1.0, n01, n10
    return float(stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue), n01, n10

def remove_words(text, ws):
    if not ws:
        return text
    pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in ws) + r")\b", re.I)
    return re.sub(r"\s+", " ", pat.sub(" ", text)).strip()

def keep_words(text, ws):
    s = set(w.lower() for w in ws)
    return " ".join(w for w in WORD.findall(text) if w.lower() in s)

def minmax_pm1(d):                      # Eq. (1) of the base paper
    v = np.array(list(d.values()), float)
    lo, hi = v.min(), v.max()
    if hi - lo < 1e-12:
        return {k: 0.0 for k in d}
    return {k: 2 * (x - lo) / (hi - lo) - 1 for k, x in d.items()}

def absmax(d):                          # sign-preserving alternative
    v = np.array(list(d.values()), float); m = np.abs(v).max()
    if m < 1e-12:
        return {k: 0.0 for k in d}
    return {k: x / m for k, x in d.items()}

# Obfuscation attacks (detector robustness)
HOMO = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с"}
def atk_homoglyph(t, r=0.15, seed=0):
    rng = random.Random(seed)
    return "".join(HOMO[c] if c in HOMO and rng.random() < r else c for c in t)
def atk_typo(t, r=0.10, seed=0):
    rng = random.Random(seed); out = []
    for w in t.split(" "):
        if len(w) > 3 and rng.random() < r:
            i = rng.randrange(len(w) - 1); w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        out.append(w)
    return " ".join(out)
def atk_zerowidth(t, r=0.10, seed=0):
    rng = random.Random(seed); out = []
    for w in t.split(" "):
        if len(w) > 3 and rng.random() < r:
            i = rng.randrange(1, len(w)); w = w[:i] + "​" + w[i:]
        out.append(w)
    return " ".join(out)
ATTACKS = {"homoglyph": atk_homoglyph, "typo": atk_typo, "zero_width": atk_zerowidth}

### Load data, duplicate / leakage audit

In [ ]:
def load_csv():
    p = CFG["csv_path"]
    if not os.path.exists(p) and DRIVE_DIR and os.path.exists(DRIVE_DIR + "/Phishing_Email.csv"):
        p = DRIVE_DIR + "/Phishing_Email.csv"
    if not os.path.exists(p):
        from google.colab import files
        up = files.upload(); p = list(up.keys())[0]
        if DRIVE_DIR:
            shutil.copy(p, DRIVE_DIR + "/Phishing_Email.csv")
    df = pd.read_csv(p); df.columns = [c.strip() for c in df.columns]
    tcol = [c for c in df.columns if "text" in c.lower()][0]
    lcol = [c for c in df.columns if "type" in c.lower() or "label" in c.lower()][0]
    df = df[[tcol, lcol]].dropna(); df.columns = ["text", "label_raw"]
    df["text"] = df["text"].map(clean)
    df = df[df.text.str.len() > 0].reset_index(drop=True)
    df["y"] = df.label_raw.astype(str).str.lower().str.contains("phish").astype(int)
    df["key"] = df.text.map(key)
    return df

raw = load_csv()
if QUICK:
    raw = pd.concat([raw[raw.y == 1].head(300), raw[raw.y == 0].head(300)]).reset_index(drop=True)
audit = dict(n_raw=len(raw), n_phish=int(raw.y.sum()), n_safe=int((1 - raw.y).sum()))
dups = raw[raw.duplicated("key", keep=False)]
audit["n_rows_in_duplicate_groups"] = len(dups)
audit["n_duplicates_removed"] = int(raw.duplicated("key").sum())
audit["n_conflicting_label_groups"] = int((dups.groupby("key").y.nunique() > 1).sum())
dedup = raw.drop_duplicates("key", keep="first").reset_index(drop=True)
audit["n_dedup"] = len(dedup); audit["n_dedup_phish"] = int(dedup.y.sum())
print(json.dumps(audit, indent=1))

def split(df, seed):
    tr, te = train_test_split(df, test_size=0.20, stratify=df.y, random_state=seed)
    tr, va = train_test_split(tr, test_size=0.125, stratify=tr.y, random_state=seed)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

### Transformer training / prediction

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_TOK = {}
def get_tok(name):
    if name not in _TOK:
        _TOK[name] = AutoTokenizer.from_pretrained(name)
    return _TOK[name]

@torch.no_grad()
def predict_proba(model, tok, texts, max_len=None, bs=64):
    max_len = max_len or CFG["max_len"]
    model.eval(); out = []
    texts = list(texts)
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt").to(dev)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
            lg = model(**enc).logits
        out.append(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def train_transformer(name, seed, tr, va):
    set_seed(seed)
    tok = get_tok(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    n_steps = CFG["epochs"] * math.ceil(len(tr) / CFG["bs"])
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / (0.1 * n_steps), max(0.0, (n_steps - s) / (0.9 * n_steps))))
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
    freq = tr.y.value_counts(normalize=True).sort_index().values
    w = torch.tensor(1.0 / (2 * freq), dtype=torch.float, device=dev)   # class-weighted loss
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    best, best_state, log = -1, None, []
    X, Y = tr.text.tolist(), tr.y.values
    for ep in range(CFG["epochs"]):
        model.train(); idx = np.random.RandomState(seed + ep).permutation(len(X)); tl = 0
        for i in range(0, len(X), CFG["bs"]):
            b = idx[i:i + CFG["bs"]]
            enc = tok([X[j] for j in b], truncation=True, max_length=CFG["max_len"],
                      padding=True, return_tensors="pt").to(dev)
            yb = torch.tensor(Y[b], device=dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                lg = model(**enc).logits
            loss = lossf(lg.float(), yb)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tl += loss.item() * len(b)
        pv = predict_proba(model, tok, va.text)
        f1 = metrics(va.y.values, (pv >= .5).astype(int), pv)["f1"]
        log.append(dict(epoch=ep + 1, train_loss=tl / len(X), val_f1=f1))
        print(name, seed, log[-1])
        if f1 > best:
            best = f1; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    return model, tok, log

### Experiments: baselines + transformers over seeds and two protocols

In [ ]:
def run_baselines(tr, va, te, seed):
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
    Xtr, Xte = vec.fit_transform(tr.text), vec.transform(te.text)
    ms = {"LR": LogisticRegression(C=10, max_iter=2000, class_weight="balanced"),
          "SVM": LinearSVC(C=1.0, class_weight="balanced"),
          "RF": RandomForestClassifier(300, n_jobs=-1, random_state=seed, class_weight="balanced")}
    res, models = {}, {}
    for k, m in ms.items():
        m.fit(Xtr, tr.y)
        p = m.predict(Xte)
        prob = m.predict_proba(Xte)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xte)
        res[k] = dict(pred=p, prob=prob, m=metrics(te.y.values, p, prob))
        models[k] = m
    return res, (vec, models)

results = []          # one row per (protocol, seed, model)
preds_store = {}      # (protocol, seed, model) -> (pred, prob)
kept = {}             # trained transformer objects for explanation (protocol B, first seed)
logs = {}
t0 = time.time()
for protocol, data in [("raw", raw), ("dedup", dedup)]:
    seeds = CFG["seeds"] if protocol == "dedup" else CFG["seeds"][:1]
    for seed in seeds:
        tr, va, te = split(data, seed)
        if protocol == "raw":
            tr_keys = set(tr.key)
            te["seen_in_train"] = te.key.isin(tr_keys)
            audit["raw_test_seen_in_train_frac_seed%d" % seed] = float(te.seen_in_train.mean())
        base, (vec, bm) = run_baselines(tr, va, te, seed)
        for k, v in base.items():
            results.append(dict(protocol=protocol, seed=seed, model=k, **v["m"]))
            preds_store[(protocol, seed, k)] = (v["pred"], v["prob"])
        if protocol == "dedup" and seed == CFG["seeds"][0]:
            kept["tfidf"] = (vec, bm["LR"])
        for name in CFG["models"]:
            if seed not in CFG["seeds_for"][name]:
                continue
            want_keep = (protocol == "dedup" and seed == CFG["seeds"][0])
            cf = ckf("train", protocol, seed, name) + ".pkl"
            mdir = ckf("model", name)
            if os.path.exists(cf) and (not want_keep or os.path.isdir(mdir)):
                rec = pickle.load(open(cf, "rb")); prob, log = rec["prob"], rec["log"]
                print("resumed", protocol, seed, name, flush=True)
                model = tok = None
                if want_keep:
                    tok = get_tok(name)
                    model = AutoModelForSequenceClassification.from_pretrained(mdir).to(dev).eval()
            else:
                model, tok, log = train_transformer(name, seed, tr, va)
                prob = predict_proba(model, tok, te.text)
                pickle.dump(dict(prob=prob, log=log), open(cf, "wb"))
                sync_drive()
                if want_keep:
                    model.save_pretrained(mdir)
                    sync_drive()
            pred = (prob >= .5).astype(int)
            results.append(dict(protocol=protocol, seed=seed, model=name,
                                **metrics(te.y.values, pred, prob)))
            preds_store[(protocol, seed, name)] = (pred, prob)
            logs[f"{protocol}_{seed}_{name}"] = log
            if protocol == "raw":      # leakage analysis: seen vs unseen test emails
                for flag in (True, False):
                    msk = te.seen_in_train.values == flag
                    if msk.sum() > 20:
                        mm = metrics(te.y.values[msk], pred[msk])
                        results.append(dict(protocol=f"raw_seen={flag}", seed=seed,
                                            model=name, n=int(msk.sum()), **mm))
            if want_keep:
                kept[name] = (model, tok, tr, va, te)
            else:
                del model
            torch.cuda.empty_cache()
        if protocol == "dedup" and seed == CFG["seeds"][0]:
            kept["split"] = (tr, va, te)
print("training+eval minutes:", (time.time() - t0) / 60)
res_df = pd.DataFrame(results)
res_df.to_csv(OUT + "/all_results.csv", index=False)
summ = (res_df[res_df.protocol == "dedup"].groupby("model")[["acc", "prec", "rec", "f1", "mcc", "fpr", "fnr", "auc"]]
        .agg(["mean", "std"]))
summ.to_csv(OUT + "/summary_dedup.csv"); print(summ.round(4))

# Bootstrap CIs and McNemar on the first dedup seed
tr0, va0, te0 = kept["split"]; y0 = te0.y.values
sig = []
first = CFG["seeds"][0]
for k in [k for k in ["LR", "SVM", "RF"] + CFG["models"] if ("dedup", first, k) in preds_store]:
    p = preds_store[("dedup", first, k)][0]
    lo, hi = bootstrap_ci(y0, p, lambda a, b: precision_recall_fscore_support(a, b, average="binary", zero_division=0)[2])
    sig.append(dict(model=k, f1=float(precision_recall_fscore_support(y0, p, average="binary")[2]), f1_lo=lo, f1_hi=hi))
pd.DataFrame(sig).to_csv(OUT + "/f1_bootstrap_ci.csv", index=False)
mc = []
best_t = max(CFG["models"], key=lambda m: res_df[(res_df.protocol == "dedup") & (res_df.model == m)].f1.mean())
for k in ["LR", "SVM", "RF"]:
    pv, n01, n10 = mcnemar(y0, preds_store[("dedup", first, best_t)][0], preds_store[("dedup", first, k)][0])
    mc.append(dict(a=best_t, b=k, p_value=pv, a_wrong_b_right=n01, a_right_b_wrong=n10))
pd.DataFrame(mc).to_csv(OUT + "/mcnemar.csv", index=False); print(pd.DataFrame(mc))

### Robustness to obfuscation (dedup split, first seed)

In [ ]:
rob = []
for name in CFG["models"]:
    if name not in kept: continue
    model, tok, *_ = kept[name]
    base_p = predict_proba(model, tok, te0.text) >= .5
    rob.append(dict(model=name, attack="none", **metrics(y0, base_p.astype(int))))
    for an, fn in ATTACKS.items():
        pt = [fn(t, seed=i) for i, t in enumerate(te0.text)]
        pr = predict_proba(model, tok, pt) >= .5
        rob.append(dict(model=name, attack=an, **metrics(y0, pr.astype(int))))
vec, lr_m = kept["tfidf"]
rob.append(dict(model="TFIDF-LR", attack="none", **metrics(y0, lr_m.predict(vec.transform(te0.text)))))
for an, fn in ATTACKS.items():
    pt = [fn(t, seed=i) for i, t in enumerate(te0.text)]
    rob.append(dict(model="TFIDF-LR", attack=an, **metrics(y0, lr_m.predict(vec.transform(pt)))))
rob_df = pd.DataFrame(rob); rob_df.to_csv(OUT + "/robustness.csv", index=False); print(rob_df[["model", "attack", "acc", "f1", "fnr"]].round(4))

### External / cross-dataset check (optional; skipped if unavailable)

In [ ]:
ext_rows = []
for dsid, cfgname in CFG["ext_datasets"]:
    try:
        from datasets import load_dataset
        d = load_dataset(dsid, cfgname, split="train") if cfgname else load_dataset(dsid, split="train")
        dfx = d.to_pandas()
        tcol = [c for c in dfx.columns if c.lower() in ("text", "email text", "email", "body", "message")][0]
        lcol = [c for c in dfx.columns if "label" in c.lower() or "type" in c.lower()][0]
        yy = dfx[lcol]
        if yy.dtype == object:
            yy = yy.astype(str).str.lower().str.contains("phish|spam").astype(int)
        dfx = pd.DataFrame(dict(text=dfx[tcol].map(clean), y=yy.astype(int))).dropna()
        dfx = dfx[dfx.text.str.len() > 0].sample(min(3000, len(dfx)), random_state=0)
        for name in CFG["models"]:
            if name not in kept: continue
            model, tok, *_ = kept[name]
            pr = (predict_proba(model, tok, dfx.text) >= .5).astype(int)
            ext_rows.append(dict(dataset=dsid, model=name, n=len(dfx), pos_rate=float(dfx.y.mean()), **metrics(dfx.y.values, pr)))
        ext_rows.append(dict(dataset=dsid, model="TFIDF-LR", n=len(dfx), pos_rate=float(dfx.y.mean()),
                             **metrics(dfx.y.values, lr_m.predict(vec.transform(dfx.text)))))
    except Exception as e:
        ext_rows.append(dict(dataset=dsid, model="SKIPPED", error=str(e)[:200]))
pd.DataFrame(ext_rows).to_csv(OUT + "/cross_dataset.csv", index=False); print(pd.DataFrame(ext_rows))

### Explanation methods: LIME, SHAP, Integrated Gradients, LITA variants

In [ ]:
from lime.lime_text import LimeTextExplainer
import shap
from captum.attr import LayerIntegratedGradients

def vocab_of(text):
    seen, out = set(), []
    for w in WORD.findall(text):
        w = w.lower()
        if w not in seen:
            seen.add(w); out.append(w)
    return out

def exp_lime(model, tok, text, vocab, seed):
    ex = LimeTextExplainer(class_names=["safe", "phish"], random_state=seed)
    def pf(xs):
        p = predict_proba(model, tok, xs)
        return np.stack([1 - p, p], 1)
    e = ex.explain_instance(text, pf,
                            num_features=len(vocab), num_samples=CFG["lime_samples"], labels=(1,))
    d = {w: 0.0 for w in vocab}
    for w, s in e.as_list(label=1):
        if w.lower() in d: d[w.lower()] += s
    return d

def exp_shap(model, tok, text, vocab):
    f = lambda xs: predict_proba(model, tok, list(xs))
    ex = shap.Explainer(f, shap.maskers.Text(r"\W+"), algorithm="partition")
    e = ex([text], max_evals=CFG["shap_evals"])
    d = {w: 0.0 for w in vocab}
    for t, v in zip(e.data[0], e.values[0]):
        t = str(t).strip().lower()
        if t in d: d[t] += float(v if np.ndim(v) == 0 else v[-1])
    return d

def exp_ig(model, tok, text, vocab):
    enc = tok(text, truncation=True, max_length=CFG["max_len"], return_offsets_mapping=True, return_tensors="pt")
    offs = enc.pop("offset_mapping")[0].numpy(); enc = {k: v.to(dev) for k, v in enc.items()}
    ids = enc["input_ids"]
    base = ids.clone(); base[:, 1:-1] = tok.pad_token_id
    emb = model.base_model.embeddings
    fwd = lambda x, am: torch.softmax(model(input_ids=x, attention_mask=am).logits.float(), -1)[:, 1]
    lig = LayerIntegratedGradients(fwd, emb)
    model.zero_grad()
    at = lig.attribute(ids, baselines=base, additional_forward_args=(enc["attention_mask"],),
                       n_steps=CFG["ig_steps"], internal_batch_size=16)
    sc = at.sum(-1)[0].detach().cpu().numpy()
    spans = [(m.start(), m.end(), m.group().lower()) for m in WORD.finditer(text)]
    d = {w: 0.0 for w in vocab}
    for (a, b), s in zip(offs, sc):
        if b <= a: continue
        for ws, we, w in spans:
            if a < we and b > ws:
                if w in d: d[w] += float(s)
                break
    return d

def lita(l, t, alpha, norm):
    nl, nt = norm(l), norm(t)
    return {w: alpha * nl[w] + (1 - alpha) * nt[w] for w in l}

def curves(model, tok, text, scores, K=10):
    """Return p_orig, p(remove top-k), p(keep only top-k) for k=1..K (top = most pro-phishing)."""
    order = [w for w, _ in sorted(scores.items(), key=lambda x: -x[1])]
    texts = [text] + [remove_words(text, order[:k]) for k in range(1, K + 1)] \
                   + [keep_words(text, order[:k]) for k in range(1, K + 1)]
    p = predict_proba(model, tok, texts)
    return p[0], p[1:K + 1], p[K + 1:]

def faith(p0, rem, keep):
    comp = float(np.mean(p0 - rem)); suff = float(np.mean(p0 - keep))
    return comp, suff

def pick_emails(te, model, tok, n, seed):
    sub = te[te.y == 1].copy()
    sub["tx"] = sub.text.map(lambda t: " ".join(t.split()[:CFG["expl_words"]]))
    sub["p"] = predict_proba(model, tok, sub.tx)
    sub = sub[sub.p >= .5]
    return sub.sample(min(n, len(sub)), random_state=seed).tx.tolist()

### Run the explanation study (resumable: progress is checkpointed every 5 emails)

In [ ]:
expl_rows, curve_store, alpha_rows, agree_rows, example = [], {}, [], [], {}
ALPHAS = [round(a, 1) for a in np.arange(0, 1.01, .1)]
METHODS = ["LIME", "SHAP", "IG", "LITA-minmax(a=.5)", "LITA-absmax(a=.5)", "LITA-absmax(tuned)", "Random"]
for name in CFG["models"]:
    if name not in kept: continue
    model, tok, tr, va, te = kept[name]
    for p_ in model.parameters(): p_.requires_grad_(False)
    ts = pick_emails(te, model, tok, CFG["n_expl_test"], 1)
    vs = pick_emails(va, model, tok, CFG["n_expl_val"], 2)

    def scores(text, seed, with_shap=True):
        vocab = vocab_of(text)
        t_ = time.time(); L = exp_lime(model, tok, text, vocab, seed); tl = time.time() - t_
        S, ts_ = None, 0.0
        if with_shap:
            t_ = time.time(); S = exp_shap(model, tok, text, vocab); ts_ = time.time() - t_
        t_ = time.time(); G = exp_ig(model, tok, text, vocab); tg = time.time() - t_
        return vocab, L, S, G, (tl, ts_, tg)

    # 1) choose alpha on validation emails (maximise mean comprehensiveness of LITA)
    vf = ckf("valcache", name) + ".pkl"
    if os.path.exists(vf):
        val_cache = pickle.load(open(vf, "rb")); print("resumed val cache", name, flush=True)
    else:
        val_cache = []
        for i, tx in enumerate(vs):
            _, L, _, G, _ = scores(tx, i, with_shap=False); val_cache.append((tx, L, G))
            if i % 10 == 0: print(name, "validation email", i + 1, "/", len(vs), flush=True)
        pickle.dump(val_cache, open(vf, "wb"))
        sync_drive()
    for norm_name, norm in [("minmax", minmax_pm1), ("absmax", absmax)]:
        for a in ALPHAS:
            c = [faith(*curves(model, tok, tx, lita(L, G, a, norm)))[0] for tx, L, G in val_cache]
            alpha_rows.append(dict(model=name, norm=norm_name, alpha=a, val_comp=float(np.mean(c))))
    adf = pd.DataFrame([r for r in alpha_rows if r["model"] == name])
    best_a = {n: float(adf[adf.norm == n].sort_values("val_comp", ascending=False).iloc[0].alpha) for n in ("minmax", "absmax")}

    # 2) evaluate all methods on test emails, checkpointing per email
    ef = ckf("expl", name) + ".pkl"
    recs = pickle.load(open(ef, "rb")) if os.path.exists(ef) else []
    start_idx = len(recs)
    if start_idx: print("resuming", name, "explanations from email", start_idx, flush=True)
    t_start = time.time()
    for i in range(start_idx, len(ts)):
        tx = ts[i]
        v, L, S, G, tm = scores(tx, 100 + i)
        rng = np.random.RandomState(1000 + i)
        cand = {"LIME": L, "SHAP": S, "IG": G,
                "LITA-minmax(a=.5)": lita(L, G, .5, minmax_pm1),
                "LITA-absmax(a=.5)": lita(L, G, .5, absmax),
                "LITA-absmax(tuned)": lita(L, G, best_a["absmax"], absmax),
                "Random": {w: rng.randn() for w in v}}
        rec = dict(tm=tm, res={}, agree=None, example=None)
        for m, sc in cand.items():
            p0, rem, keep = curves(model, tok, tx, sc)
            c, s_ = faith(p0, rem, keep)
            rec["res"][m] = (c, s_, p0 - rem, p0 - keep)
        ws = list(L)
        if len(ws) > 3:
            a_, b_ = np.array([L[w] for w in ws]), np.array([G[w] for w in ws])
            top = np.argsort(-np.abs(a_))[:10]
            ta, tb = set(np.argsort(-a_)[:5]), set(np.argsort(-b_)[:5])
            rec["agree"] = dict(spearman_lime_ig=float(stats.spearmanr(a_, b_)[0]),
                                sign_disagree_top10=float(np.mean(np.sign(a_[top]) != np.sign(b_[top]))),
                                top5_jaccard=len(ta & tb) / max(len(ta | tb), 1),
                                spearman_lime_shap=float(stats.spearmanr(a_, [S[w] for w in ws])[0]),
                                spearman_shap_ig=float(stats.spearmanr([S[w] for w in ws], b_)[0]))
        if i == 0:
            rec["example"] = dict(text=tx, L=L, S=S, G=G, H=cand["LITA-absmax(tuned)"], Hmm=cand["LITA-minmax(a=.5)"])
        recs.append(rec)
        if (i + 1) % 5 == 0 or i + 1 == len(ts):
            pickle.dump(recs, open(ef, "wb"))
            sync_drive()
            done = i + 1 - start_idx; el = time.time() - t_start
            eta = el / done * (len(ts) - i - 1) / 60
            print(f"{name}: {i + 1}/{len(ts)} emails, ~{eta:.0f} min left", flush=True)

    acc = {m: dict(comp=[r["res"][m][0] for r in recs], suff=[r["res"][m][1] for r in recs],
                   rem=[r["res"][m][2] for r in recs], keep=[r["res"][m][3] for r in recs]) for m in METHODS}
    times = [r["tm"] for r in recs]
    for r in recs:
        if r["agree"] is not None: agree_rows.append(dict(model=name, **r["agree"]))
    if recs and recs[0]["example"] is not None: example[name] = recs[0]["example"]
    for m, d in acc.items():
        expl_rows.append(dict(model=name, method=m, n=len(recs), comp_mean=np.mean(d["comp"]), comp_sd=np.std(d["comp"]),
                              suff_mean=np.mean(d["suff"]), suff_sd=np.std(d["suff"]), alpha_tuned_absmax=best_a["absmax"]))
        curve_store[(name, m)] = (np.mean(d["rem"], 0), np.mean(d["keep"], 0))
    tt = np.mean(times, 0)
    expl_rows.append(dict(model=name, method="_runtime_sec(LIME,SHAP,IG)", n=len(recs),
                          comp_mean=tt[0], comp_sd=tt[1], suff_mean=tt[2], suff_sd=0, alpha_tuned_absmax=np.nan))
    for m in ["LIME", "SHAP", "IG", "LITA-minmax(a=.5)"]:
        try:
            w = stats.wilcoxon(acc["LITA-absmax(tuned)"]["comp"], acc[m]["comp"])
            expl_rows.append(dict(model=name, method=f"wilcoxon_tunedLITA_vs_{m}", n=len(recs), comp_mean=w.pvalue))
        except Exception:
            pass
ex_df = pd.DataFrame(expl_rows); ex_df.to_csv(OUT + "/explanation_faithfulness.csv", index=False)
pd.DataFrame(alpha_rows).to_csv(OUT + "/alpha_sweep.csv", index=False)
ag = pd.DataFrame(agree_rows); ag.to_csv(OUT + "/explanation_agreement_raw.csv", index=False)
ag.groupby("model").agg(["mean", "std"]).to_csv(OUT + "/explanation_agreement.csv")
print(ex_df.round(4)); print(ag.groupby("model").mean().round(3))

### Figures

In [ ]:
plt.rcParams.update({"font.size": 9, "figure.dpi": 150})
d = res_df[res_df.protocol == "dedup"]
order = [m for m in ["LR", "SVM", "RF"] + CFG["models"] if m in set(d.model)]
fig, ax = plt.subplots(figsize=(3.4, 2.4))
mu = [d[d.model == m].f1.mean() for m in order]; sd = [d[d.model == m].f1.std() if (d.model == m).sum() > 1 else 0 for m in order]
ax.bar(order, mu, yerr=sd, capsize=3, color="#4c78a8"); ax.set_ylim(max(0.8, min(mu) - .05), 1.0)
ax.set_ylabel("Phishing F1 (mean ± sd)"); plt.xticks(rotation=25); plt.tight_layout()
plt.savefig(OUT + "/figures/fig_f1.png", dpi=300); plt.close()

fig, axs = plt.subplots(1, 2, figsize=(7, 2.5))
for (name, m), (rem, keep) in curve_store.items():
    if name != CFG["models"][0]: continue
    axs[0].plot(range(1, 11), rem, label=m); axs[1].plot(range(1, 11), keep, label=m)
axs[0].set_title("Comprehensiveness (higher = better)"); axs[1].set_title("Sufficiency gap (lower = better)")
for a in axs: a.set_xlabel("k (top words)")
axs[0].set_ylabel("Drop in P(phishing)"); axs[1].legend(fontsize=6); plt.tight_layout()
plt.savefig(OUT + "/figures/fig_faithfulness_curves.png", dpi=300); plt.close()

adf = pd.DataFrame(alpha_rows)
fig, ax = plt.subplots(figsize=(3.4, 2.4))
for (mm, nn), g in adf.groupby(["model", "norm"]):
    if mm == CFG["models"][0]: ax.plot(g.alpha, g.val_comp, marker="o", ms=3, label=nn)
ax.set_xlabel(r"$\alpha$ (weight on LIME)"); ax.set_ylabel("Val. comprehensiveness"); ax.legend(); plt.tight_layout()
plt.savefig(OUT + "/figures/fig_alpha_sweep.png", dpi=300); plt.close()

fig, ax = plt.subplots(figsize=(3.4, 2.4)); w_ = .25
rb = rob_df[rob_df.model.isin(["TFIDF-LR", CFG["models"][0]])]
for j, m in enumerate(rb.model.unique()):
    g = rb[rb.model == m]; ax.bar(np.arange(len(g)) + j * w_, g.f1, w_, label=m)
ax.set_xticks(np.arange(len(g)) + w_ / 2); ax.set_xticklabels(g.attack, rotation=20); ax.set_ylabel("Phishing F1"); ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig(OUT + "/figures/fig_robustness.png", dpi=300); plt.close()

nm = CFG["models"][0]
if nm in example:
    e = example[nm]; ws = sorted(e["L"], key=lambda w: -abs(e["H"][w]))[:12]
    fig, axs = plt.subplots(1, 4, figsize=(7, 2.6), sharey=True)
    for a, (t, dd) in zip(axs, [("LIME", e["L"]), ("SHAP", e["S"]), ("IG", e["G"]), ("LITA (tuned)", e["H"])]):
        v = np.array([dd[w] for w in ws]); v = v / (np.abs(v).max() + 1e-12)
        a.barh(range(len(ws)), v, color=["#d62728" if x > 0 else "#1f77b4" for x in v]); a.set_title(t, fontsize=8)
    axs[0].set_yticks(range(len(ws))); axs[0].set_yticklabels(ws, fontsize=7); axs[0].invert_yaxis()
    plt.tight_layout(); plt.savefig(OUT + "/figures/fig_example_attributions.png", dpi=300); plt.close()
    json.dump({k: (v if isinstance(v, str) else {w: float(s) for w, s in v.items()}) for k, v in e.items()},
              open(OUT + "/example_attributions.json", "w"), indent=1)

### Save metadata and zip

In [ ]:
sync_drive("final")
print(sorted(os.listdir(CKPT)))
import transformers, sklearn
meta = dict(audit=audit, config={k: v for k, v in CFG.items() if k != "ext_datasets"}, quick=QUICK,
            gpu=torch.cuda.get_device_name(0) if dev == "cuda" else "cpu",
            versions=dict(torch=torch.__version__, transformers=transformers.__version__, sklearn=sklearn.__version__,
                          shap=shap.__version__), training_logs=logs, best_transformer=best_t)
json.dump(meta, open(OUT + "/meta.json", "w"), indent=1, default=str)
with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for r, _, fs in os.walk(OUT):
        for f in fs: z.write(os.path.join(r, f))
if DRIVE_DIR: shutil.copy("results.zip", os.path.dirname(OUT) + "/results.zip")
print("Done. Download results.zip (a copy is also in your Google Drive, folder phish_xai)")
try:
    from google.colab import files; files.download("results.zip")
except Exception:
    pass
sync_drive("after meta")
print("Drive folders now:", os.listdir(os.path.dirname(CKPT)))
